# PPO Atari — Frame-Stacking Study (automatic, 1 GPU, in series)

Runs the whole study automatically: `FRAME_STACK ∈ {1,2,4,8}` × `GAME ∈ {Pong, Breakout}`
× `SEED ∈ {0,1,2}` = **24 runs**, one after another on a single GPU.

Why one GPU in series (not 2 in parallel): Atari is **CPU-bound** (the game simulation
runs on CPU). Two runs sharing the CPU each slow down ~3×, so parallel is actually
*slower* overall. One run at a time keeps the whole CPU for it → faster total time.

Key fix vs the first version: **Atari reward clipping** (`sign(reward)` → +1/0/−1),
the standard from Mnih et al. (2015). The previous `REWARD_SCALE=0.01` (kept from the
VizDoom code) shrank Pong's ±1 rewards to ±0.01 and stopped learning.

Resume: a finished run writes a `.done` file and is skipped if you re-run the loop,
so after a disconnect you just run the loop cell again.


## 0. Install dependencies

In [1]:
!pip install -q "gymnasium[atari]" ale-py imageio pandas

## 1. All the code (PPO from the professor, Atari env, reward clipping fixed)

Everything is defined as functions so the loop in the next cell can call `train_one(...)`
for each config. The PPO algorithm is identical to the professor's VizDoom code; only the
environment and the reward handling changed (marked `# CHANGED` / `# FIX`).

In [2]:
import os, csv, numpy as np, torch
import torch.nn as nn, torch.optim as optim
from torch.distributions import Categorical
from collections import deque
import gymnasium as gym
import ale_py
gym.register_envs(ale_py)

# -------- fixed hyperparameters (the professor's) --------
HORIZON=512; N_ACTORS=8; EPOCHS_PER_UPDATE=3; BATCH_SIZE=32*8
LR=2.5e-4; CLIP_EPS=0.2; GAMMA=0.99; GAE_LAMBDA=0.95; VF_COEF=0.5; ENT_COEF=0.01
IMG_SIZE=84; CHANNELS_PER_FRAME=1; FRAME_SKIP=4
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUTDIR = "/kaggle/working"
print("device:", device)

class AtariEnv:
    """Atari screen -> grayscale 84x84 -> stack N frames. Same interface as prof's VizDoomEnv."""
    def __init__(self, env_id, frame_stack, seed=0):
        from gymnasium.wrappers import AtariPreprocessing, FrameStackObservation
        env = gym.make(env_id, frameskip=1)
        env = AtariPreprocessing(env, frame_skip=FRAME_SKIP, screen_size=IMG_SIZE,
                                 grayscale_obs=True, scale_obs=False)
        env = FrameStackObservation(env, stack_size=frame_stack)
        self.env = env; self._seed = seed
        self.n_actions = env.action_space.n
    def _fmt(self, o): return np.asarray(o, dtype=np.uint8)
    def reset(self):
        o,_ = self.env.reset(seed=self._seed); self._seed += 1; return self._fmt(o)
    def step(self, a):
        o,r,term,trunc,_ = self.env.step(a)
        r = float(np.sign(r))          # FIX: Atari reward clipping +1/0/-1 (Mnih et al. 2015)
        return self._fmt(o), r, bool(term or trunc)
    def close(self): self.env.close()

class ActorCritic(nn.Module):   # UNCHANGED from prof
    def __init__(self, in_channels, n_actions):
        super().__init__()
        self.body = nn.Sequential(
            nn.Conv2d(in_channels,32,8,4), nn.ReLU(),
            nn.Conv2d(32,64,4,2), nn.ReLU(),
            nn.Conv2d(64,64,3,1), nn.ReLU(), nn.Flatten())
        with torch.no_grad():
            n_features = self.body(torch.zeros(1,in_channels,IMG_SIZE,IMG_SIZE)).shape[1]
        self.hidden = nn.Sequential(nn.Linear(n_features,256), nn.ReLU())
        self.actor = nn.Linear(256,n_actions); self.critic = nn.Linear(256,1)
    def forward(self,x):
        x = x.float()/255.0; h = self.hidden(self.body(x))
        return self.actor(h), self.critic(h).squeeze(-1)
    def act(self,x):
        logits,value = self.forward(x); dist = Categorical(logits=logits)
        a = dist.sample(); return a, dist.log_prob(a), value
    def evaluate(self,x,a):
        logits,value = self.forward(x); dist = Categorical(logits=logits)
        return dist.log_prob(a), dist.entropy(), value

def compute_gae(rewards, values, dones, last_value):   # UNCHANGED from prof
    adv = torch.zeros_like(rewards); gae = torch.zeros_like(last_value)
    values = torch.cat([values, last_value.unsqueeze(0)], dim=0)
    for t in reversed(range(rewards.shape[0])):
        nt = 1.0 - dones[t]
        delta = rewards[t] + GAMMA*values[t+1]*nt - values[t]
        gae = delta + GAMMA*GAE_LAMBDA*nt*gae
        adv[t] = gae
    return adv, adv + values[:-1]

def train_one(game, frame_stack, seed, total_steps):
    run = f"{game}_fs{frame_stack}_seed{seed}"
    hist = os.path.join(OUTDIR, f"ppo_history_{run}.csv")
    donef = os.path.join(OUTDIR, f"ppo_history_{run}.done")
    save = os.path.join(OUTDIR, f"ppo_{run}.pt")
    if os.path.exists(donef):
        print(f"skip {run} (already done)"); return
    np.random.seed(seed); torch.manual_seed(seed)
    env_id = f"ALE/{game}-v5"
    envs = [AtariEnv(env_id, frame_stack, seed=seed*100+i) for i in range(N_ACTORS)]
    n_actions = envs[0].n_actions
    in_channels = frame_stack*CHANNELS_PER_FRAME
    agent = ActorCritic(in_channels, n_actions).to(device)
    opt = optim.Adam(agent.parameters(), lr=LR)
    obs = np.stack([e.reset() for e in envs])
    ep_r=np.zeros(N_ACTORS); ep_s=np.zeros(N_ACTORS,dtype=int)
    er=deque(maxlen=20); el=deque(maxlen=20); step=0; upd=0
    with open(hist,"w",newline="") as f:
        csv.writer(f).writerow(["step","mean_reward","mean_episode_length","policy_loss","value_loss","entropy","kl"])
    print(f"=== START {run}  env={env_id}  in_channels={in_channels} ===", flush=True)
    while step < total_steps:
        alpha=max(1.0-step/total_steps,0.0)
        for g in opt.param_groups: g["lr"]=LR*alpha
        clip=CLIP_EPS*alpha
        ob=torch.zeros((HORIZON,N_ACTORS,in_channels,IMG_SIZE,IMG_SIZE),dtype=torch.uint8,device=device)
        ac=torch.zeros((HORIZON,N_ACTORS),dtype=torch.long,device=device)
        lp=torch.zeros((HORIZON,N_ACTORS),device=device); rw=torch.zeros((HORIZON,N_ACTORS),device=device)
        dn=torch.zeros((HORIZON,N_ACTORS),device=device); vl=torch.zeros((HORIZON,N_ACTORS),device=device)
        for t in range(HORIZON):
            ot=torch.from_numpy(obs).to(device)
            with torch.no_grad(): a,logp,v = agent.act(ot)
            acts=a.cpu().numpy(); rews=np.zeros(N_ACTORS,np.float32); dones=np.zeros(N_ACTORS,np.float32)
            nxt=np.empty_like(obs)
            for i,env in enumerate(envs):
                o,r,d=env.step(int(acts[i])); rews[i],dones[i]=r,float(d)
                ep_r[i]+=r; ep_s[i]+=1
                if d:
                    er.append(ep_r[i]); el.append(ep_s[i]); ep_r[i],ep_s[i]=0.0,0; o=env.reset()
                nxt[i]=o
            ob[t]=ot; ac[t]=a; lp[t]=logp
            rw[t]=torch.from_numpy(rews).to(device)      # FIX: no scaling, rewards are already clipped
            dn[t]=torch.from_numpy(dones).to(device); vl[t]=v; obs=nxt; step+=N_ACTORS
        with torch.no_grad(): _,lastv = agent.forward(torch.from_numpy(obs).to(device))
        adv,ret = compute_gae(rw,vl,dn,lastv)
        bo=ob.reshape(-1,in_channels,IMG_SIZE,IMG_SIZE); ba=ac.reshape(-1); blp=lp.reshape(-1)
        bad=adv.reshape(-1); bre=ret.reshape(-1); n=bo.shape[0]; pls,vls,ents,kls=[],[],[],[]
        for _ in range(EPOCHS_PER_UPDATE):
            idx=torch.randperm(n,device=device)
            for s in range(0,n,BATCH_SIZE):
                mb=idx[s:s+BATCH_SIZE]
                nlp,ent,nv=agent.evaluate(bo[mb],ba[mb])
                ratio=(nlp-blp[mb]).exp(); a_=bad[mb]; a_=(a_-a_.mean())/(a_.std()+1e-8)
                s1=ratio*a_; s2=torch.clamp(ratio,1-clip,1+clip)*a_
                pl=-torch.min(s1,s2).mean(); vlo=(nv-bre[mb]).pow(2).mean(); em=ent.mean()
                loss=pl+VF_COEF*vlo-ENT_COEF*em
                opt.zero_grad(); loss.backward()
                nn.utils.clip_grad_norm_(agent.parameters(),0.5); opt.step()
                with torch.no_grad(): kl=(blp[mb]-nlp).mean().item()
                pls.append(pl.item()); vls.append(vlo.item()); ents.append(em.item()); kls.append(kl)
        mr=np.mean(er) if er else float("nan"); ml=np.mean(el) if el else float("nan")
        with open(hist,"a",newline="") as f:
            csv.writer(f).writerow([step,mr,ml,np.mean(pls),np.mean(vls),np.mean(ents),np.mean(kls)])
        upd+=1
        if upd % 20 == 0:
            print(f"[{run}] step={step}/{total_steps} mean_reward={mr:.2f} value_loss={np.mean(vls):.4f}", flush=True)
        if upd % 50 == 0: torch.save(agent.state_dict(), save)
    torch.save(agent.state_dict(), save)
    open(donef,"w").write("done")
    for e in envs: e.close()
    print(f"=== DONE {run}  final mean_reward={mr:.2f} ===", flush=True)

device: cuda


## 2. Run the whole study

- Leave `TOTAL_STEPS = 500_000` for a **first test** (confirms the reward now rises and
  `value_loss` is non-zero). Watch the first run's `mean_reward`.
- Once confirmed, set `TOTAL_STEPS = 2_000_000` and run this cell again — finished runs
  are skipped, so it continues with the full budget.

Order: all frame settings and seeds for Pong first, then Breakout.

In [3]:
import shutil, glob, os

for src in glob.glob("/kaggle/input/**/ppo_history_*", recursive=True):
    dst = os.path.join("/kaggle/working", os.path.basename(src))
    if not os.path.exists(dst):
        shutil.copy(src, dst)
print("files:", len(glob.glob("/kaggle/working/ppo_history_*.done")), "runs already done")


files: 30 runs already done


In [4]:
import itertools

GAMES  = ["Pong"]
FRAMES = [1, 2, 4, 8, 16]
SEEDS  = [0, 1, 2]
TOTAL_STEPS = 2_000_000


for game, fs, seed in itertools.product(GAMES, FRAMES, SEEDS):
    train_one(game, fs, seed, TOTAL_STEPS)

print("ALL RUNS COMPLETE")

A.L.E: Arcade Learning Environment (version 0.12.1+8a8fafb)
[Powered by Stella]


=== START Pong_fs1_seed0  env=ALE/Pong-v5  in_channels=1 ===
[Pong_fs1_seed0] step=81920/2000000 mean_reward=-20.00 value_loss=0.1202
[Pong_fs1_seed0] step=163840/2000000 mean_reward=-20.70 value_loss=0.0948
[Pong_fs1_seed0] step=245760/2000000 mean_reward=-20.10 value_loss=0.1075
[Pong_fs1_seed0] step=327680/2000000 mean_reward=-20.45 value_loss=0.0639
[Pong_fs1_seed0] step=409600/2000000 mean_reward=-20.20 value_loss=0.0701
[Pong_fs1_seed0] step=491520/2000000 mean_reward=-20.15 value_loss=0.0717
[Pong_fs1_seed0] step=573440/2000000 mean_reward=-19.80 value_loss=0.0705
[Pong_fs1_seed0] step=655360/2000000 mean_reward=-19.70 value_loss=0.0823
[Pong_fs1_seed0] step=737280/2000000 mean_reward=-19.80 value_loss=0.0707
[Pong_fs1_seed0] step=819200/2000000 mean_reward=-19.10 value_loss=0.0745
[Pong_fs1_seed0] step=901120/2000000 mean_reward=-18.35 value_loss=0.0642
[Pong_fs1_seed0] step=983040/2000000 mean_reward=-17.70 value_loss=0.0766
[Pong_fs1_seed0] step=1064960/2000000 mean_reward=-1

## 3. Quick check — list finished runs and their final reward

In [5]:
import glob, pandas as pd, os
for f in sorted(glob.glob("/kaggle/working/ppo_history_*.csv")):
    df = pd.read_csv(f)
    v = df["mean_reward"].dropna()
    vlast = v.iloc[-1] if len(v) else float("nan")
    print(f"{os.path.basename(f):38s} updates={len(df):4d} final_reward={vlast:.2f}")

ppo_history_Breakout_fs16_seed0.csv    updates= 489 final_reward=15.80
ppo_history_Breakout_fs16_seed1.csv    updates= 489 final_reward=9.40
ppo_history_Breakout_fs16_seed2.csv    updates= 489 final_reward=14.15
ppo_history_Breakout_fs1_seed0.csv     updates= 489 final_reward=9.50
ppo_history_Breakout_fs1_seed1.csv     updates= 489 final_reward=7.85
ppo_history_Breakout_fs1_seed2.csv     updates= 489 final_reward=6.85
ppo_history_Breakout_fs2_seed0.csv     updates= 489 final_reward=7.40
ppo_history_Breakout_fs2_seed1.csv     updates= 489 final_reward=10.15
ppo_history_Breakout_fs2_seed2.csv     updates= 489 final_reward=14.30
ppo_history_Breakout_fs4_seed0.csv     updates= 489 final_reward=12.05
ppo_history_Breakout_fs4_seed1.csv     updates= 489 final_reward=14.95
ppo_history_Breakout_fs4_seed2.csv     updates= 489 final_reward=17.15
ppo_history_Breakout_fs8_seed0.csv     updates= 489 final_reward=13.85
ppo_history_Breakout_fs8_seed1.csv     updates= 489 final_reward=17.90
ppo_history